# Historical House Price Trends — All Visualizations

This notebook consolidates all visualizations produced for the ECE 143 group project presentation.

**Contents:**
1. **State-Level Choropleth Animation** — animated U.S. map of year-over-year price changes (1975–2024)
2. **Fastest-Growing Valuations** — state bar charts and county choropleths for 2008 crisis, COVID era, and 5-year trends
3. **ARIMA HPI Forecasting** — 10-year forecast for all 51 states with evaluation, choropleths, and error analysis

**Prerequisites:** Run `python src/data_cleaning.py` first to generate the cleaned CSVs in `output/`.

---
## Setup

In [ ]:
import sys
import warnings
import json
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)

from data_cleaning import prepare_state_hpi
from prediction import split_train_test, run_arima_all_states, compute_metrics
from plot_utils import (
    plot_state_forecast, plot_forecast_grid,
    plot_forecast_choropleth, plot_forecast_choropleth_animated,
    plot_metrics_bar,
)

print("Imports loaded.")

---
# Part 1: State-Level Choropleth Animation

An animated choropleth of annual HPI percentage change by state (1975–2024). Uses a **Red–Yellow–Green** color scale (range: -20% to +20%):
- **Red** = declining prices
- **Yellow** = flat / modest growth
- **Green** = strong appreciation

Key patterns: the 2006–2009 housing crash, the post-COVID boom (2020–2022), and regional volatility differences.

In [ ]:
df_state_growth = pd.read_csv("../output/state_growth_rates.csv")
df_state_growth.head(10)

In [ ]:
tables = pd.read_html('https://developers.google.com/public-data/docs/canonical/states_csv')
state_coords = tables[0]
state_coords.columns = ['state', 'latitude', 'longitude', 'name']

df_choropleth = df_state_growth.merge(
    state_coords[['state', 'latitude', 'longitude']],
    left_on='Abbreviation', right_on='state',
)

fig = px.choropleth(
    df_choropleth,
    locations='Abbreviation',
    locationmode='USA-states',
    color='Annual Change (%)',
    color_continuous_scale='RdYlGn',
    range_color=(-20, 20),
    scope='usa',
    animation_frame='Year',
    labels={'Annual Change (%)': 'Annual Change (%)'},
    title='Annual House Price Change by State',
)

coords = df_choropleth.drop_duplicates(subset='Abbreviation')
label_trace = go.Scattergeo(
    locationmode='USA-states',
    lon=coords['longitude'],
    lat=coords['latitude'],
    text=coords['Abbreviation'],
    mode='text',
    textfont=dict(size=8, color='white'),
    showlegend=False,
    hoverinfo='skip',
)
fig.add_trace(label_trace)
for frame in fig.frames:
    frame.data = (*frame.data, label_trace)

fig.update_layout(margin={'r': 0, 't': 50, 'l': 0, 'b': 0})
fig.show()

---
# Part 2: Fastest-Growing Valuations

State bar charts and county choropleths examining HPI changes during key periods:
1. **2008–2009 Financial Crisis**
2. **COVID-19 Era (2020–2021)**
3. **Fastest-growing states (2020–2024)** — 5-year averages
4. **Fastest-growing counties (2020–2024)** — 5-year averages

In [ ]:
df_state = pd.read_csv("../output/state_growth_rates.csv")
df_state["Annual Change (%)"] = pd.to_numeric(df_state["Annual Change (%)"], errors="coerce")
df_state = df_state.dropna(subset=["Abbreviation", "Year", "Annual Change (%)"])

df_county = pd.read_csv("../output/county_growth_rates.csv", dtype={"FIPS code": str})
df_county["Annual Change (%)"] = pd.to_numeric(df_county["Annual Change (%)"], errors="coerce")
df_county = df_county.dropna(subset=["FIPS code", "Year", "Annual Change (%)"])

df_state_fast = pd.read_csv("../output/state_fastest_growth.csv")
df_county_fast = pd.read_csv("../output/county_fastest_growth.csv", dtype={"FIPS code": str})

with urlopen("https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json") as response:
    counties_geojson = json.load(response)

### 2008–2009 Financial Crisis

State bar chart and county choropleth for 2008 and 2009.

In [ ]:
for year in [2008, 2009]:
    s = df_state[df_state["Year"] == year].sort_values("Annual Change (%)", ascending=False)
    fig = px.bar(
        s,
        x="Annual Change (%)",
        y="Abbreviation",
        orientation="h",
        title=f"State level: Annual HPI change ({year})",
        labels={"Annual Change (%)": "Annual change (%)", "Abbreviation": "State"},
        color="Annual Change (%)",
        color_continuous_scale="RdYlGn",
        range_color=(-15, 25),
    )
    fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
    fig.show()

for year in [2008, 2009]:
    c = df_county[df_county["Year"] == year]
    fig = px.choropleth_map(
        c,
        geojson=counties_geojson,
        locations="FIPS code",
        featureidkey="id",
        color="Annual Change (%)",
        hover_name="County",
        hover_data={"State": True, "Annual Change (%)": ":.2f"},
        color_continuous_scale="RdYlGn",
        range_color=(-15, 25),
        map_style="carto-positron",
        zoom=3,
        center={"lat": 37.0902, "lon": -95.7129},
        opacity=0.5,
        title=f"County level: Annual HPI change ({year})",
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    fig.show()

### COVID-19 Era (2020–2021)

State bar chart and county choropleth for 2020 and 2021 (pandemic and post-pandemic housing surge).

In [ ]:
for year in [2020, 2021]:
    s = df_state[df_state["Year"] == year].sort_values("Annual Change (%)", ascending=False)
    fig = px.bar(
        s,
        x="Annual Change (%)",
        y="Abbreviation",
        orientation="h",
        title=f"State level: Annual HPI change ({year})",
        labels={"Annual Change (%)": "Annual change (%)", "Abbreviation": "State"},
        color="Annual Change (%)",
        color_continuous_scale="RdYlGn",
        range_color=(-15, 25),
    )
    fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
    fig.show()

for year in [2020, 2021]:
    c = df_county[df_county["Year"] == year]
    fig = px.choropleth_map(
        c,
        geojson=counties_geojson,
        locations="FIPS code",
        featureidkey="id",
        color="Annual Change (%)",
        hover_name="County",
        hover_data={"State": True, "Annual Change (%)": ":.2f"},
        color_continuous_scale="RdYlGn",
        range_color=(-15, 25),
        map_style="carto-positron",
        zoom=3,
        center={"lat": 37.0902, "lon": -95.7129},
        opacity=0.5,
        title=f"County level: Annual HPI change ({year})",
    )
    fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
    fig.show()

### Fastest-Growing States (2020–2024)

Top 15 and bottom 15 states by average annual HPI change over the last 5 years, plus a scatter comparing 5-year vs 10-year growth.

In [ ]:
top_15 = df_state_fast.sort_values("Rank by 5yr Growth").head(15)
bottom_15 = df_state_fast.sort_values("Rank by 5yr Growth").tail(15)

fig = px.bar(
    top_15,
    x="Avg Annual Change (5yr) %",
    y="Abbreviation",
    orientation="h",
    title="Top 15 states by avg annual HPI change (2020 - 2024)",
    labels={"Avg Annual Change (5yr) %": "Avg annual change (5yr) %", "Abbreviation": "State"},
    color="Avg Annual Change (5yr) %",
    color_continuous_scale="RdYlGn",
    range_color=(0, 12),
)
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()

fig = px.bar(
    bottom_15,
    x="Avg Annual Change (5yr) %",
    y="Abbreviation",
    orientation="h",
    title="Bottom 15 states by avg annual HPI change (2020 - 2024)",
    labels={"Avg Annual Change (5yr) %": "Avg annual change (5yr) %", "Abbreviation": "State"},
    color="Avg Annual Change (5yr) %",
    color_continuous_scale="RdYlGn",
    range_color=(0, 12),
)
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()

fig = px.scatter(
    df_state_fast,
    x="Avg Annual Change (5yr) %",
    y="Avg Annual Change (10yr) %",
    text="Abbreviation",
    title="5-year vs 10-year avg annual HPI change by state",
    labels={
        "Avg Annual Change (5yr) %": "Avg annual change (5yr) %",
        "Avg Annual Change (10yr) %": "Avg annual change (10yr) %",
    },
    color="Avg Annual Change (5yr) %",
    color_continuous_scale="RdYlGn",
    range_color=(0, 12),
)
fig.update_traces(textposition="top center")
fig.update_layout(showlegend=False)
fig.show()

### Fastest-Growing Counties (2020–2024)

County choropleth colored by 5-year average annual HPI change, plus top 20 and bottom 20 counties nationally.

In [ ]:
cf = df_county_fast.dropna(subset=["Avg Annual Change (5yr) %"])

fig = px.choropleth_map(
    cf,
    geojson=counties_geojson,
    locations="FIPS code",
    featureidkey="id",
    color="Avg Annual Change (5yr) %",
    hover_name="County",
    hover_data={"State": True, "Avg Annual Change (5yr) %": ":.2f"},
    color_continuous_scale="RdYlGn",
    range_color=(-5, 20),
    map_style="carto-positron",
    zoom=3,
    center={"lat": 37.0902, "lon": -95.7129},
    opacity=0.5,
    title="County level: Avg annual HPI change (2020 - 2024)",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

top_20 = cf.sort_values("Rank National (5yr)").head(20)
top_20 = top_20.copy()
top_20["Label"] = top_20["County"] + ", " + top_20["State"]

fig = px.bar(
    top_20,
    x="Avg Annual Change (5yr) %",
    y="Label",
    orientation="h",
    title="Top 20 counties by avg annual HPI change (2020 - 2024)",
    labels={"Avg Annual Change (5yr) %": "Avg annual change (5yr) %", "Label": "County"},
    color="Avg Annual Change (5yr) %",
    color_continuous_scale="RdYlGn",
    range_color=(0, 25),
)
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()

bottom_20 = cf.sort_values("Rank National (5yr)").tail(20)
bottom_20 = bottom_20.copy()
bottom_20["Label"] = bottom_20["County"] + ", " + bottom_20["State"]

fig = px.bar(
    bottom_20,
    x="Avg Annual Change (5yr) %",
    y="Label",
    orientation="h",
    title="Bottom 20 counties by avg annual HPI change (2020 - 2024)",
    labels={"Avg Annual Change (5yr) %": "Avg annual change (5yr) %", "Label": "County"},
    color="Avg Annual Change (5yr) %",
    color_continuous_scale="RdYlGn",
    range_color=(-20, 10),
)
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()

---
# Part 3: ARIMA HPI Forecasting

We use **ARIMA** (AutoRegressive Integrated Moving Average) to forecast HPI 10 years into the future for all 51 U.S. states. `pmdarima.auto_arima` automatically selects the best (p, d, q) order per state via AIC minimization.

- **Training**: Fit on historical HPI data per state
- **Evaluation**: 5-year holdout (2020–2024) with MAE, RMSE, and MAPE
- **Forecast**: 10-year ahead predictions (2025–2034) with 95% confidence intervals

### Load & Explore HPI Data

In [ ]:
raw_dir = project_root / "data" / "raw"
df = prepare_state_hpi(raw_dir)

print(f"Shape: {df.shape}")
print(f"States: {df['Abbreviation'].nunique()}")
print(f"Year range: {df['Year'].min()} \u2013 {df['Year'].max()}")
df.head(10)

In [ ]:
latest_year = df["Year"].max()
latest = df[df["Year"] == latest_year].sort_values("HPI", ascending=False)
print(f"\nHPI in {latest_year} \u2014 Top 5 and Bottom 5:")
print(latest[["Abbreviation", "State", "HPI"]].head())
print("...")
print(latest[["Abbreviation", "State", "HPI"]].tail())

### Fit ARIMA for All States

This takes ~1–3 minutes.

In [ ]:
%%time
arima_forecast, arima_eval, arima_models = run_arima_all_states(
    df, forecast_years=10, test_years=5
)
print(f"ARIMA forecasts: {len(arima_forecast)} rows")
print(f"ARIMA eval: {len(arima_eval)} rows")

### Evaluate ARIMA on 2020–2024 Holdout

In [ ]:
metrics = compute_metrics(arima_eval)

print("=== ARIMA Metrics (Best 10 by MAPE) ===")
print(metrics.head(10).to_string(index=False))
print(f"\nMedian MAPE: {metrics['MAPE'].median():.2f}%")
print(f"Mean MAPE: {metrics['MAPE'].mean():.2f}%")
print(f"States with MAPE < 15%: {(metrics['MAPE'] < 15).sum()}/51")

print("\n=== Worst 10 by MAPE ===")
print(metrics.tail(10).to_string(index=False))

### Single State Forecast (California)

In [ ]:
fig = plot_state_forecast("CA", df, arima_forecast, eval_df=arima_eval)
fig.show()

### Multi-State Forecast Grid

In [ ]:
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=["CA", "TX", "NY", "FL", "IL", "WA"],
)
fig.show()

### Forecasts for All 51 States

In [ ]:
all_states = sorted(df["Abbreviation"].unique())
fig = plot_forecast_grid(
    df, arima_forecast, eval_df=arima_eval,
    states=all_states,
)
fig.show()

### Choropleth Map — Predicted HPI in 2030

In [ ]:
fig = plot_forecast_choropleth(arima_forecast, 2030)
fig.show()

### Animated Choropleth — History + Forecast

In [ ]:
fig = plot_forecast_choropleth_animated(df, arima_forecast, history_years=5)
fig.show()

### MAPE by State Bar Chart

In [ ]:
fig = plot_metrics_bar(metrics, metric="MAPE", top_n=20)
fig.show()

### ARIMA Model Orders Selected per State

`auto_arima` selects a different (p, d, q) order for each state. Let's examine what orders were chosen.

In [ ]:
order_rows = []
for state, model in arima_models.items():
    p, d, q = model.order
    order_rows.append({"Abbreviation": state, "p": p, "d": d, "q": q,
                       "Order": f"({p},{d},{q})", "AIC": round(model.aic(), 2)})
orders_df = pd.DataFrame(order_rows).sort_values("Abbreviation")

print("ARIMA Orders Selected by auto_arima:\n")
print(orders_df.to_string(index=False))

print(f"\n--- Order Distribution ---")
print(orders_df["Order"].value_counts().to_string())
print(f"\nMost common d (differencing): {orders_df['d'].mode().iloc[0]} "
      f"({(orders_df['d'] == orders_df['d'].mode().iloc[0]).sum()}/51 states)")
print(f"Mean AIC: {orders_df['AIC'].mean():.1f}")

### Regional Performance Analysis

MAPE grouped by U.S. Census region.

In [ ]:
regions = {
    "Northeast": ["CT", "ME", "MA", "NH", "RI", "VT", "NJ", "NY", "PA"],
    "Midwest": ["IL", "IN", "MI", "OH", "WI", "IA", "KS", "MN", "MO", "NE", "ND", "SD"],
    "South": ["DE", "FL", "GA", "MD", "NC", "SC", "VA", "DC", "WV",
              "AL", "KY", "MS", "TN", "AR", "LA", "OK", "TX"],
    "West": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY",
             "AK", "CA", "HI", "OR", "WA"],
}
state_to_region = {s: r for r, states in regions.items() for s in states}
metrics_with_region = metrics.copy()
metrics_with_region["Region"] = metrics_with_region["Abbreviation"].map(state_to_region)

print("=== MAPE by Region ===\n")
region_stats = metrics_with_region.groupby("Region")["MAPE"].agg(["mean", "median", "min", "max", "count"])
region_stats.columns = ["Mean MAPE", "Median MAPE", "Best MAPE", "Worst MAPE", "N States"]
region_stats = region_stats.sort_values("Median MAPE")
print(region_stats.to_string())

print("\n\n=== Best & Worst State per Region ===\n")
for region in region_stats.index:
    r = metrics_with_region[metrics_with_region["Region"] == region].sort_values("MAPE")
    best = r.iloc[0]
    worst = r.iloc[-1]
    print(f"{region}:")
    print(f"  Best:  {best['Abbreviation']} ({best['State']}) \u2014 MAPE {best['MAPE']:.2f}%")
    print(f"  Worst: {worst['Abbreviation']} ({worst['State']}) \u2014 MAPE {worst['MAPE']:.2f}%")

### Error Analysis: Forecast Bias

The 2020–2024 holdout includes the COVID-era housing boom. Let's check whether ARIMA systematically underestimated the surge.

In [ ]:
eval_merged = arima_eval.merge(
    df[["Abbreviation", "Year", "HPI"]], on=["Abbreviation", "Year"], how="left"
)
eval_merged["Signed_Error"] = eval_merged["Predicted"] - eval_merged["HPI"]
eval_merged["Pct_Error"] = (eval_merged["Signed_Error"] / eval_merged["HPI"] * 100)

print("=== Year-by-Year Forecast Bias (Predicted \u2212 Actual) ===\n")
yearly = eval_merged.groupby("Year").agg(
    Mean_Signed_Error=("Signed_Error", "mean"),
    Median_Signed_Error=("Signed_Error", "median"),
    Mean_Pct_Error=("Pct_Error", "mean"),
    MAPE=("Pct_Error", lambda x: x.abs().mean()),
    Underpredicted=("Signed_Error", lambda x: (x < 0).sum()),
).reset_index()
yearly["Year"] = yearly["Year"].astype(int)

for _, r in yearly.iterrows():
    direction = "UNDER" if r["Mean_Signed_Error"] < 0 else "OVER"
    print(f"  {r['Year']}:  Mean Error: {r['Mean_Signed_Error']:+7.1f}  "
          f"({r['Mean_Pct_Error']:+5.1f}%)  |  MAPE: {r['MAPE']:5.1f}%  |  "
          f"{direction}-predicted {int(r['Underpredicted'])}/51 states")

print(f"\nOverall: ARIMA underpredicted in {int(yearly['Underpredicted'].mean())}/51 states on average.")

growth_rates = []
for state in df["Abbreviation"].unique():
    state_df = df[df["Abbreviation"] == state].sort_values("Year")
    hpi_2019 = state_df[state_df["Year"] == 2019]["HPI"].values
    hpi_2024 = state_df[state_df["Year"] == 2024]["HPI"].values
    if len(hpi_2019) > 0 and len(hpi_2024) > 0:
        pct_growth = (hpi_2024[0] - hpi_2019[0]) / hpi_2019[0] * 100
        growth_rates.append({"Abbreviation": state, "Growth_Pct": round(pct_growth, 1)})
growth_df = pd.DataFrame(growth_rates)
scatter_df = metrics.merge(growth_df, on="Abbreviation")

slope, intercept, r_value, _, _ = stats.linregress(scatter_df["Growth_Pct"], scatter_df["MAPE"])

fig = px.scatter(
    scatter_df, x="Growth_Pct", y="MAPE", text="Abbreviation",
    labels={"Growth_Pct": "HPI Growth 2019\u21922024 (%)", "MAPE": "MAPE (%)"},
    title=f"Forecast Error vs Housing Boom Intensity (R\u00b2 = {r_value**2:.2f})",
)
fig.update_traces(textposition="top center", marker=dict(size=8))
x_range = np.linspace(scatter_df["Growth_Pct"].min(), scatter_df["Growth_Pct"].max(), 100)
fig.add_scatter(x=x_range, y=intercept + slope * x_range, mode="lines",
                name=f"OLS (slope={slope:.2f})", line=dict(dash="dash", color="red"))
fig.update_layout(height=500, width=800)
fig.show()

print(f"Correlation: r = {r_value:.2f}, R\u00b2 = {r_value**2:.2f}")

### Forecast Outlook: Predicted HPI Rankings in 2034

In [ ]:
print("=" * 65)
print("FORECAST OUTLOOK: PREDICTED HPI RANKINGS IN 2034")
print("=" * 65)

forecast_2034 = arima_forecast[arima_forecast["Year"] == 2034].copy()
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
outlook = forecast_2034.merge(latest_actual, on="Abbreviation")
outlook["Growth_2024_2034_Pct"] = ((outlook["Predicted"] - outlook["HPI_2024"]) / outlook["HPI_2024"] * 100).round(1)

print("\nTop 10 states by predicted HPI in 2034:")
top = outlook.sort_values("Predicted", ascending=False).head(10)
for _, r in top.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print("\nBottom 10 states by predicted HPI in 2034:")
bot = outlook.sort_values("Predicted").head(10)
for _, r in bot.iterrows():
    print(f"  {r['Abbreviation']:2s} {r['State']:22s}  HPI: {r['Predicted']:8.1f}  (2024: {r['HPI_2024']:.1f}, +{r['Growth_2024_2034_Pct']:.1f}%)")

print(f"\nHighest predicted growth:  {outlook.sort_values('Growth_2024_2034_Pct', ascending=False).iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].max():.1f}%)")
print(f"Lowest predicted growth:   {outlook.sort_values('Growth_2024_2034_Pct').iloc[0]['Abbreviation']} (+{outlook['Growth_2024_2034_Pct'].min():.1f}%)")
print(f"Median predicted growth:   +{outlook['Growth_2024_2034_Pct'].median():.1f}%")

### Confidence Interval Analysis

In [ ]:
print("=" * 65)
print("CONFIDENCE INTERVAL ANALYSIS")
print("=" * 65)
print("\nWider 95% CI = more uncertainty in the forecast.\n")

ci_analysis = arima_forecast.copy()
ci_analysis["CI_Width"] = ci_analysis["CI_Upper"] - ci_analysis["CI_Lower"]

ci_2034 = ci_analysis[ci_analysis["Year"] == 2034].sort_values("CI_Width", ascending=False)
latest_actual = df[df["Year"] == 2024][["Abbreviation", "HPI"]].rename(columns={"HPI": "HPI_2024"})
ci_2034 = ci_2034.merge(latest_actual, on="Abbreviation")
ci_2034["CI_Width_Pct"] = (ci_2034["CI_Width"] / ci_2034["HPI_2024"] * 100).round(1)

print("States with widest 95% CI in 2034 (most uncertain):")
for _, r in ci_2034.head(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

print("\nStates with narrowest 95% CI in 2034 (most confident):")
for _, r in ci_2034.tail(10).iterrows():
    print(f"  {r['Abbreviation']:2s}  CI Width: {r['CI_Width']:8.1f}  ({r['CI_Width_Pct']:5.1f}% of 2024 HPI)")

print("\nAverage CI width by forecast year:")
for year in sorted(ci_analysis["Year"].unique()):
    avg_width = ci_analysis[ci_analysis["Year"] == year]["CI_Width"].mean()
    print(f"  {int(year)}: {avg_width:.1f}")